# 📖 How2Sign Continuous Training Pipeline

**Purpose:** Train a Continuous Sign Language Recognition (CSLR) model using the How2Sign dataset.
This pipeline loads pre-extracted OpenPose keypoints from `.json` files and trains a Seq2Seq BiLSTM to output sequences of words.

In [ ]:
# ============================================================
# 1. SETUP & CONFIGURATION
# ============================================================
import os
import numpy as np
import pandas as pd
import json
import tensorflow as tf
from pathlib import Path

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f'GPU mode: {len(gpus)} GPU(s), mixed_float16')
else:
    print('CPU mode')


## 2. Dataset Paths
Pointing to the downloaded CSV and JSON artifacts.

In [ ]:
# ============================================================
# 2. DATASET PATHS  (Kaggle – How2Sign-keypoints dataset)
# ============================================================
BASE_DIR = Path('/kaggle/input/how2sign-keypoints')

CSV_TRAIN = BASE_DIR / 'how2sign_realigned_train.csv'
CSV_VAL   = BASE_DIR / 'how2sign_realigned_val.csv'
CSV_TEST  = BASE_DIR / 'how2sign_realigned_test.csv'

# JSON frames live one folder deeper than the split root
JSON_DIR_TRAIN = BASE_DIR / 'train_2D_keypoints' / 'openpose_output' / 'json'
JSON_DIR_VAL   = BASE_DIR / 'val_2D_keypoints'   / 'openpose_output' / 'json'
JSON_DIR_TEST  = BASE_DIR / 'test_2D_keypoints'  / 'openpose_output' / 'json'

for label, p in [('BASE_DIR', BASE_DIR),
                 ('CSV_TRAIN', CSV_TRAIN), ('CSV_VAL', CSV_VAL), ('CSV_TEST', CSV_TEST),
                 ('JSON_TRAIN', JSON_DIR_TRAIN), ('JSON_VAL', JSON_DIR_VAL), ('JSON_TEST', JSON_DIR_TEST)]:
    print(f"{label:12s} exists={p.exists()}  → {p}")


## 3. Custom Data Generator
This generator reads the OpenPose JSON files batch-by-batch to prevent out-of-memory errors.

In [ ]:
# ============================================================
# 3. CUSTOM DATA GENERATOR (OpenPose JSON → 78-dim feature vector)
# ============================================================
from tensorflow.keras.utils import Sequence

# ---------------------------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------------------------
NUM_BODY_KPS  = 25   # OpenPose body_25 model
NUM_HAND_KPS  = 21   # each hand
# 78-dim = body (x,y) × 25 + left-hand wrist (x,y) × 4
#        = 50 + 28  — we use (x,y) of 14 selected hand keypoints
# Simpler split actually used here:
#   pose_keypoints_2d  → 25 × 3 = 75 values  (x, y, conf per joint)
#   face center (nose) → 3 more values        → total 78
NOSE_IDX = 0   # index in pose_keypoints_2d for nose keypoint (body_25)

def convert_openpose_to_78dim(frame_data: dict) -> np.ndarray:
    """
    Parse one OpenPose frame JSON dict and return a (78,) float32 array.

    OpenPose frame JSON structure:
        { "version": ...,
          "people": [
            { "pose_keypoints_2d":  [x0,y0,c0, x1,y1,c1, ...],  # 25 kps × 3 = 75
              "face_keypoints_2d":  [...],
              "hand_left_keypoints_2d":  [...],
              "hand_right_keypoints_2d": [...] }
          ]
        }
    We extract the 75 body values + the 3 face-nose values (or zeros) = 78.
    """
    feat = np.zeros(78, dtype=np.float32)
    people = frame_data.get('people', [])
    if not people:
        return feat

    person = people[0]

    # --- body pose: 25 kps × 3 = 75 values ---
    pose_raw = person.get('pose_keypoints_2d', [])
    pose_arr = np.array(pose_raw, dtype=np.float32)
    n_body = min(len(pose_arr), 75)
    feat[:n_body] = pose_arr[:n_body]

    # --- face: use the nose keypoint (index 0 in face_keypoints_2d) ---
    face_raw = person.get('face_keypoints_2d', [])
    if len(face_raw) >= 3:
        feat[75:78] = np.array(face_raw[:3], dtype=np.float32)

    return feat


def load_sentence_frames(sentence_name: str, json_dir: Path) -> list:
    """
    Return a list of (78,) arrays – one per frame – for the given sentence.
    Frame JSON files are stored at:
        <json_dir>/<sentence_name>/<sentence_name>_XXXXXX_keypoints.json
    """
    sentence_dir = json_dir / sentence_name
    if not sentence_dir.exists():
        return []

    frame_files = sorted(sentence_dir.glob('*.json'))
    frames = []
    for fp in frame_files:
        with fp.open('r') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
        frames.append(convert_openpose_to_78dim(data))
    return frames


# ---------------------------------------------------------------------------
# Generator
# ---------------------------------------------------------------------------
class How2SignGenerator(Sequence):
    """Keras Sequence generator for How2Sign continuous sign recognition."""

    # Expected CSV columns (How2Sign realigned format)
    COL_NAME     = 'SENTENCE_NAME'
    COL_SENTENCE = 'SENTENCE'

    def __init__(self, csv_path: Path, json_dir: Path,
                 tokenizer=None,
                 batch_size: int = 8,
                 sequence_length: int = 150,
                 num_features: int = 78):

        if not csv_path.exists():
            print(f'⚠️  CSV not found: {csv_path}')
            self.df = pd.DataFrame()
        else:
            self.df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
            # Fallback: try comma-separated
            if self.COL_NAME not in self.df.columns:
                self.df = pd.read_csv(csv_path, on_bad_lines='skip')
            print(f'✅ Loaded {len(self.df):,} rows from {csv_path.name}')
            print(f'   Columns: {list(self.df.columns)}')

        self.json_dir        = json_dir
        self.tokenizer       = tokenizer
        self.batch_size      = batch_size
        self.sequence_length = sequence_length
        self.num_features    = num_features

    # ---- helpers -----------------------------------------------------------

    def _pad_frames(self, frames: list) -> np.ndarray:
        """Pad / truncate frame list to (sequence_length, num_features)."""
        out = np.zeros((self.sequence_length, self.num_features), dtype=np.float32)
        T = min(len(frames), self.sequence_length)
        if T > 0:
            out[:T] = np.stack(frames[:T])
        return out

    # ---- Sequence interface ------------------------------------------------

    def __len__(self):
        if len(self.df) == 0:
            return 0
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]

        X = np.zeros((len(batch), self.sequence_length, self.num_features),
                     dtype=np.float32)

        for i, (_, row) in enumerate(batch.iterrows()):
            name   = str(row.get(self.COL_NAME, ''))
            frames = load_sentence_frames(name, self.json_dir)
            X[i]   = self._pad_frames(frames)

        # Labels: return raw sentences for now (tokenizer can be wired later)
        sentences = batch.get(self.COL_SENTENCE,
                              pd.Series([''] * len(batch))).fillna('').tolist()
        return X, sentences


# ---------------------------------------------------------------------------
# Smoke-test: instantiate generators and peek at one batch
# ---------------------------------------------------------------------------
train_gen = How2SignGenerator(CSV_TRAIN, JSON_DIR_TRAIN, batch_size=4)
val_gen   = How2SignGenerator(CSV_VAL,   JSON_DIR_VAL,   batch_size=4)
test_gen  = How2SignGenerator(CSV_TEST,  JSON_DIR_TEST,  batch_size=4)

print(f'\nTrain batches : {len(train_gen)}')
print(f'Val   batches : {len(val_gen)}')
print(f'Test  batches : {len(test_gen)}')

if len(train_gen) > 0:
    X_sample, Y_sample = train_gen[0]
    print(f'\nSample batch X shape : {X_sample.shape}')
    print(f'Sample batch Y (first): {Y_sample[0]}')


## 4. Continuous Model Architecture (Seq2Seq)

In [ ]:
# ============================================================
# 4. CONTINUOUS SEQ2SEQ ARCHITECTURE
# ============================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout, BatchNormalization, SpatialDropout1D, Masking, TimeDistributed

vocab_size = 5000 

model_continuous = Sequential([
    Input(shape=(None, 78)), 
    Masking(mask_value=0.0),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    TimeDistributed(Dense(vocab_size, activation='softmax'))
])

model_continuous.summary()
